# Excel



## Setup


In [1]:
%%capture pkg_output
%%bash
apt-get update
apt-get install -y tree jq


In [2]:
import requests
import pandas
import pandas as pd
import duckdb
from google.colab import userdata
import os


## Fetching an Excel file


In [3]:
url = "https://www2.census.gov/programs-surveys/acs/summary_file/2025/table-based-SF/documentation/ACS2025_Table_Shells.xlsx"
base_file = url.split("/")[-1].split(".")[0]

url, base_file


('https://www2.census.gov/programs-surveys/acs/summary_file/2025/table-based-SF/documentation/ACS2025_Table_Shells.xlsx',
 'ACS2025_Table_Shells')

### Get the sheet names using Pandas


In [4]:
wb_dict = pd.read_excel( url, sheet_name=None, dtype=str)
wb_dict.keys()


dict_keys(['2025'])

### Preview the data in Pandas


In [5]:
wb_dict["2025"]


,Table ID,Line,UniqueID,Stub,Data Release
0,NaN,NaN,NaN,NaN,NaN
1,B01001,,NaN,Sex by Age,"1,5"
2,B01001,,NaN,Universe: Total population,NaN
3,B01001,1,B01001_001,Total:,NaN
4,B01001,2,B01001_002,Male:,NaN
...,...,...,...,...,...
41761,B99283,1.5,NaN,Allocated:,NaN
41762,B99283,2,B99283_002,Dial-up,NaN
41763,B99283,3,B99283_003,"Broadband such as cable, fiberoptic, or DSL, s...",NaN
41764,B99283,4,B99283_004,Cellular data plan,NaN


## Load the data using DuckDB


In [6]:
query = (f"""
  SELECT *
  FROM read_xlsx('{url}')
  ;
""")

print(query)

duckdb.query( query ).df()



  SELECT *
  FROM read_xlsx('https://www2.census.gov/programs-surveys/acs/summary_file/2025/table-based-SF/documentation/ACS2025_Table_Shells.xlsx')
  ;



,Table ID,Line,UniqueID,Stub,Data Release


In [7]:
query = (f"""
  SELECT *
  FROM read_xlsx('{url}', sheet = '2025')
  ;
""")

print(query)

duckdb.query( query ).df()



  SELECT *
  FROM read_xlsx('https://www2.census.gov/programs-surveys/acs/summary_file/2025/table-based-SF/documentation/ACS2025_Table_Shells.xlsx', sheet = '2025')
  ;



,Table ID,Line,UniqueID,Stub,Data Release


In [8]:
query = (f"""
  SELECT *
  FROM read_xlsx('{url}', sheet = '2025', stop_at_empty = false)
  ;
""")

print(query)

duckdb.query( query ).df()



  SELECT *
  FROM read_xlsx('https://www2.census.gov/programs-surveys/acs/summary_file/2025/table-based-SF/documentation/ACS2025_Table_Shells.xlsx', sheet = '2025', stop_at_empty = false)
  ;



InvalidInputException: Invalid Input Error: read_xlsx: Failed to parse cell 'A3': Could not convert string 'B01001' to DOUBLE

In [9]:
query = (f"""
  SELECT *
  FROM read_xlsx('{url}', sheet = '2025', stop_at_empty = false, header = false)
  ;
""")

print(query)

duckdb.query( query ).df()



  SELECT *
  FROM read_xlsx('https://www2.census.gov/programs-surveys/acs/summary_file/2025/table-based-SF/documentation/ACS2025_Table_Shells.xlsx', sheet = '2025', stop_at_empty = false, header = false)
  ;



,A1,B1,C1,D1,E1
0,Table ID,Line,UniqueID,Stub,Data Release
1,None,None,None,None,None
2,B01001,,None,Sex by Age,"1,5"
3,B01001,,None,Universe: Total population,None
4,B01001,1,B01001_001,Total:,None
...,...,...,...,...,...
41762,B99283,1.5,None,Allocated:,None
41763,B99283,2,B99283_002,Dial-up,None
41764,B99283,3,B99283_003,"Broadband such as cable, fiberoptic, or DSL, s...",None
41765,B99283,4,B99283_004,Cellular data plan,None


DuckDB is not always the optimal tool, especially with very untidy data.


## Use pandas


In [10]:
df = wb_dict["2025"]
df


,Table ID,Line,UniqueID,Stub,Data Release
0,NaN,NaN,NaN,NaN,NaN
1,B01001,,NaN,Sex by Age,"1,5"
2,B01001,,NaN,Universe: Total population,NaN
3,B01001,1,B01001_001,Total:,NaN
4,B01001,2,B01001_002,Male:,NaN
...,...,...,...,...,...
41761,B99283,1.5,NaN,Allocated:,NaN
41762,B99283,2,B99283_002,Dial-up,NaN
41763,B99283,3,B99283_003,"Broadband such as cable, fiberoptic, or DSL, s...",NaN
41764,B99283,4,B99283_004,Cellular data plan,NaN


In [11]:
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 41766 entries, 0 to 41765
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   Table ID      40580 non-null  object
 1   Line          39312 non-null  object
 2   UniqueID      37188 non-null  object
 3   Stub          40307 non-null  object
 4   Data Release  1461 non-null   object
dtypes: object(5)
memory usage: 1.6+ MB


In [12]:
filter = df["Table ID"].notnull()
df[ filter ].astype('string').to_parquet(f"{base_file}.parquet", index=False)


In [13]:
!ls -la

total 388
drwxr-xr-x 1 root root   4096 Nov 18 18:29 .
drwxr-xr-x 1 root root   4096 Nov 18 18:17 ..
-rw-r--r-- 1 root root 379192 Nov 18 18:29 ACS2025_Table_Shells.parquet
drwxr-xr-x 4 root root   4096 Oct 26 22:18 .config
drwxr-xr-x 1 root root   4096 Nov 17 15:58 sample_data


In [14]:
duckdb.query(f'''
  select *
  from "{base_file}.parquet"
  limit 10
''')

┌──────────┬─────────┬────────────┬─────────────────────────────┬──────────────┐
│ Table ID │  Line   │  UniqueID  │            Stub             │ Data Release │
│ varchar  │ varchar │  varchar   │           varchar           │   varchar    │
├──────────┼─────────┼────────────┼─────────────────────────────┼──────────────┤
│ B01001   │         │ NULL       │ Sex by Age                  │ 1,5          │
│ B01001   │         │ NULL       │ Universe:  Total population │ NULL         │
│ B01001   │ 1       │ B01001_001 │ Total:                      │ NULL         │
│ B01001   │ 2       │ B01001_002 │ Male:                       │ NULL         │
│ B01001   │ 3       │ B01001_003 │ Under 5 years               │ NULL         │
│ B01001   │ 4       │ B01001_004 │ 5 to 9 years                │ NULL         │
│ B01001   │ 5       │ B01001_005 │ 10 to 14 years              │ NULL         │
│ B01001   │ 6       │ B01001_006 │ 15 to 17 years              │ NULL         │
│ B01001   │ 7       │ B0100

## Push to Hugging Face

### Configure settings and authenticate to Hugging Face


In [15]:
hf_account = os.environ["HF_ACCOUNT"] = userdata.get('hf_account')
hf_account


'rwcitek'

In [16]:
hf_repo = os.environ["HF_REPO"] = "DUC-2026"
hf_repo


'DUC-2026'

In [17]:
hf_file = os.environ["HF_FILE"] = f"{base_file}.parquet"
hf_file


'ACS2025_Table_Shells.parquet'

In [18]:
# Retrieve the token from Colab's secret manager
_ = hf_token = os.environ["HF_TOKEN"] = userdata.get('hf_token')
f"{_[:3]} ... {_[-4:]}"


'hf_ ... dQPb'

In [19]:
# Log in automatically without an interactive prompt
! hf auth login --token ${HF_TOKEN}


Hint: A new version of huggingface_hub (2.4.0) is available! You are using version 1.33.0.
To update, run: hf update
Hint: The `hf-cli` skill is not installed. Run `hf skills add -g` to teach your AI agents how to use the `hf` CLI.
The token has not been saved to the git credentials helper. Pass `add_to_git_credential=True` in this function directly or `--add-to-git-credential` if using via `hf`CLI if you want to set the git credential as well.
Token is valid (permission: fineGrained).
The token `Colab` has been saved to /root/.cache/huggingface/stored_tokens
Your token has been saved to /root/.cache/huggingface/token
Login successful.
Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


### Push the parquet file to Hugging Face


In [20]:
%%capture hf_upload
%%bash

TARGET_REPO="${HF_ACCOUNT}/${HF_REPO}"
LOCAL_SOURCE="./${HF_FILE}"
REMOTE_DESTINATION="./${HF_FILE}"

# Comment out the echo to make this live
echo DRY-RUN: \
hf upload \
  --type dataset \
  "${TARGET_REPO}" \
  "${LOCAL_SOURCE}" \
  "${REMOTE_DESTINATION}"


In [21]:
print(hf_upload.stdout)


DRY-RUN: hf upload --type dataset rwcitek/DUC-2026 ./ACS2025_Table_Shells.parquet ./ACS2025_Table_Shells.parquet



### Query the parquet on Hugging Face using DuckDB


In [22]:
# Construct the Hugging Face URL for your dataset file
hf_url = f"https://huggingface.co/datasets/{hf_account}/{hf_repo}/resolve/main/{hf_file}"
hf_url


'https://huggingface.co/datasets/rwcitek/DUC-2026/resolve/main/ACS2025_Table_Shells.parquet'

In [23]:
# Query the remote parquet
query = (f'''
  select count(1) as count
  from "{hf_url}"
  ;
''')
print(query)
duckdb.sql(query).df()



  select count(1) as count
  from "https://huggingface.co/datasets/rwcitek/DUC-2026/resolve/main/ACS2025_Table_Shells.parquet"
  ;



,count
0,40580


In [24]:
# Query the remote parquet
query = (f'''
  describe
  select *
  from "{hf_url}"
  ;
''')
print(query)
duckdb.sql(query).df()



  describe
  select *
  from "https://huggingface.co/datasets/rwcitek/DUC-2026/resolve/main/ACS2025_Table_Shells.parquet"
  ;



,column_name,column_type,null,key,default,extra
0,Table ID,VARCHAR,YES,None,None,None
1,Line,VARCHAR,YES,None,None,None
2,UniqueID,VARCHAR,YES,None,None,None
3,Stub,VARCHAR,YES,None,None,None
4,Data Release,VARCHAR,YES,None,None,None
